# Data loading and preprocessing

## Dataset loading

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

# For type hinting, which is good practice
from torch.utils.data import Dataset
from typing import Tuple

def load_fashion_data() -> Tuple[Dataset, Dataset, Dataset]:
    """
    Loads the Fashion-MNIST dataset and splits the data into train, val, test.

    This function loads the Fashion-MNIST training and test sets from
    torchvision. It then splits the original training set into a new,
    smaller training set and a validation set (90/10 ratio).

    Returns:
        Tuple[Dataset, Dataset, Dataset]: A tuple containing the training,
        validation, and test datasets as torch.utils.data.Dataset objects.
    """
    # Use a basic transform to convert images to tensors.
    # More complex transforms will be applied later in the custom Dataset class.
    # converts a PIL Image or numpy.ndarray to tensor.
    transform = transforms.ToTensor()

    # 1. Load the full original training data
    full_train_dataset = datasets.FashionMNIST(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    # 2. Load the test data
    test_dataset = datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    # 3. Calculate split sizes
    dataset_size = len(full_train_dataset)
    train_size = int(0.9 * dataset_size)
    val_size = dataset_size - train_size

    # 4. Split the full training dataset into training and validation subsets
    # Using a fixed generator seed for reproducibility of the split
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    return train_dataset, val_dataset, test_dataset

# # Example of how to call the function and verify the output
# if __name__ == '__main__':
#     train_data, val_data, test_data = load_fashion_data()

#     print(f"Fashion-MNIST Dataset Summary:")
#     print(f"  - Total Original Training Images: {len(train_data) + len(val_data)}")
#     print(f"  - Split Training Set Size:      {len(train_data)}")
#     print(f"  - Split Validation Set Size:    {len(val_data)}")
#     print(f"  - Test Set Size:                {len(test_data)}")

## Custom Dataset Class

In [3]:
import torch
from torch.utils.data import Dataset, Subset
from torchvision import transforms
from typing import Tuple, Any

# Pre-computed mean and std for Fashion-MNIST (1 channel)
FASHION_MNIST_MEAN = (0.2860,)
FASHION_MNIST_STD = (0.3530,)